# Day 8 — Solution: Vectors

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["SPY", "TLT", "GLD"], start="2015-01-01")
else:
    px = synthetic_prices(n_days=2000, n_assets=3, seed=19, corr=0.3)
    px.columns = ["SPY", "TLT", "GLD"]
rets = px.pct_change().dropna()

## E1 — by hand

$0.5(0.012) + 0.3(-0.004) + 0.2(0.003) = 0.006 - 0.0012 + 0.0006 = 0.0054$
— the portfolio's return that day (0.54%).

In [ ]:
w = np.array([0.5, 0.3, 0.2]); r_day = np.array([0.012, -0.004, 0.003])
print(w @ r_day)

## E2 — asserts and the one-line backtest primitive

In [ ]:
w = np.array([0.5, 0.3, 0.2])
assert abs(w.sum() - 1) < 1e-12                       # fully invested
w_neutral = np.array([0.5, -0.2, -0.3])
assert abs(w_neutral.sum()) < 1e-12                   # dollar-neutral

port = rets @ w
(1 + port).cumprod().plot(title="Growth of $1: 50/30/20 portfolio")
plt.show()

## E3 — centered cosine = correlation

In [ ]:
centered = rets - rets.mean()
norms = np.linalg.norm(centered.values, axis=0)
cos = (centered.values.T @ centered.values) / np.outer(norms, norms)
print(np.round(cos, 3))
print(np.round(rets.corr().values, 3))

Identical (to floating point): **correlation is the cosine of the angle
between centered return vectors.** Two assets with ρ = 0.98 point in nearly
the same direction; ρ = 0, perpendicular; ρ = −1, opposite. This geometric
picture makes covariance-matrix facts (day 9-10) intuitive rather than
algebraic.

## E4 — a long-short portfolio

In [ ]:
cum = (1 + rets).prod()
best, worst = cum.idxmax(), cum.idxmin()
w_ls = pd.Series(0.0, index=rets.columns)
w_ls[best], w_ls[worst] = 0.5, -0.5
assert abs(w_ls.sum()) < 1e-12

pnl = rets @ w_ls
print(f"long {best} / short {worst}: vol {pnl.std() * np.sqrt(252):.2%}")

Sum of weights = 0 (dollar-neutral) yet the P&L stream has substantial
volatility: net *exposure* is zero, but *risk* is not — each leg carries
its own variance, and the covariance between legs (day 10's Σ) determines
what remains. Dollar-neutral ≠ risk-neutral.

## E5 — why "same asset" arbitrage isn't

With ρ = 0.98, the centered return vectors sit ~11.5° apart (arccos 0.98).
The difference vector (long A, short B) has small but *nonzero* norm — that
norm is the position's risk. Small × leverage = real money: at 10×
leverage, a "0.2%-ish" daily tracking noise becomes 2% daily swings on
capital, and the distribution's tails (module 03) plus financing costs eat
the theoretical convergence. Correlation ≈ 1 is a statement about an
*angle*, not an identity.